# **Aprendizaje por refuerzos** - FrozenLake y LunarLander (Gymnasium)

## Tarea: Implementar Agentes Q-Learning y DQN

### Objetivos:
1. Implementar el algoritmo Q-Learning
2. Implementar el algoritmo DQN
3. Entrenar y evaluar ambos agentes
4. Comparar el rendimiento de ambos enfoques


In [1]:
# Instalar paquetes requeridos
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

%pip install swig matplotlib gymnasium torch pygame


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importar las bibliotecas
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import pygame
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from collections import deque, namedtuple
import random


/home/hezek/.pyenv/versions/3.13.6/lib/python3.13/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


La siguiente celda permite ejecutar un juego de Frozen Lake *determinista* para jugar con el teclado.

Utilize las teclas de dirección (flechas) o asdw para comandar al agente.


In [3]:
def jugar_frozen_lake(env):
    env.reset()
    
    print("Controles:")
    print("W - Arriba")
    print("S - Abajo") 
    print("A - Izquierda")
    print("D - Derecha")
    print("Q - Salir")
    print("Presione cualquier tecla para empezar...")
    
    pygame.init()
    pygame.display.set_caption("FrozenLake - Juego Interactivo")
    
    clock = pygame.time.Clock()
    ejecutando = True
    
    while ejecutando:
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                ejecutando = False
            elif event.type == pygame.KEYDOWN:
                if event.key == pygame.K_q or event.key == pygame.K_ESCAPE:
                    ejecutando = False
                elif event.key == pygame.K_w or event.key == pygame.K_UP:
                    accion = 3  # Arriba
                elif event.key == pygame.K_s or event.key == pygame.K_DOWN:
                    accion = 1  # Abajo
                elif event.key == pygame.K_a or event.key == pygame.K_LEFT:
                    accion = 0  # Izquierda
                elif event.key == pygame.K_d or event.key == pygame.K_RIGHT:
                    accion = 2  # Derecha
                else:
                    continue
                
                observacion, recompensa, terminado, truncado, info = env.step(accion)
                print(f"Acción: {accion}, Recompensa: {recompensa}, Terminado: {terminado}")
                
                if terminado or truncado:
                    print(f"¡Episodio terminado! Recompensa final: {recompensa}")
                    pygame.time.wait(500)
                    env.reset()
        
        clock.tick(60)
    
    pygame.quit()
    env.close()

env = gym.make('FrozenLake-v1', render_mode='human', is_slippery=False)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)


La siguiente celda permite jugar al juego no determinista.

In [4]:
env = gym.make('FrozenLake-v1', map_name="4x4", render_mode='human', is_slippery=True)
# Descomente la línea de abajo para jugar interactivamente
#jugar_frozen_lake(env)

La siguiente clase define la interfaz de los agentes que utilizaremos para jugar al Frozen Lake.


In [5]:
from abc import ABC, abstractmethod

class Agente(ABC):
    
    @abstractmethod
    def elegir_accion(self, estado):
        """Elige una acción dada una observación."""
        pass
    
    @abstractmethod
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Aprende de la experiencia."""
        pass

class AgenteAleatorio(Agente):
    """Agente aleatorio que elige acciones al azar."""
    
    def __init__(self, espacio_acciones):
        # Se guarda el espacio de acciones para poder elegir acciones al azar
        self.espacio_acciones = espacio_acciones
    
    def elegir_accion(self, estado):
        return self.espacio_acciones.sample()
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        pass  # El agente aleatorio no aprende

# Probar el AgenteAleatorio
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
estado, _ = env.reset()
accion = agente_aleatorio.elegir_accion(estado)
print(f"✓ AgenteAleatorio creado y probado. Acción: {accion}")
env.close()


✓ AgenteAleatorio creado y probado. Acción: 1


La siguiente celda define una función para evaluar el desempeño de un agente dado.

In [6]:
# Función de Evaluación de Agentes
def evaluar_agente(agente, env, num_episodios=1000):
    """
    Evalúa el rendimiento de un agente a lo largo de múltiples episodios.
    
    Args:
        agente: El agente a evaluar
        env: El entorno
        num_episodios: Número de episodios a ejecutar
    
    Returns:
        dict: Resultados de la evaluación
    """
    recompensas_totales = []
    victorias = 0
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        
        while True:
            accion = agente.elegir_accion(estado)
            estado, recompensa, terminado, truncado, _ = env.step(accion)
            recompensa_total += recompensa
            
            if terminado or truncado:
                break
        
        recompensas_totales.append(recompensa_total)
        if recompensa_total > 0:
            victorias += 1
    
    return {
        'recompensas_totales': recompensas_totales,
        'victorias': victorias,
        'tasa_victorias': victorias / num_episodios,
        'recompensa_promedio': np.mean(recompensas_totales),
        'desv_estandar': np.std(recompensas_totales)
    }

def imprimir_resultados_evaluacion(resultados, nombre_agente):
    """Imprime los resultados de evaluación de forma formateada."""
    print(f"\n{nombre_agente} - Resultados de Evaluación:")
    print(f"Tasa de Victorias: {resultados['tasa_victorias']:.1%}")
    print(f"Recompensa Promedio: {resultados['recompensa_promedio']:.3f}")
    print(f"Desviación Estándar: {resultados['desv_estandar']:.3f}")
    print(f"Total de Victorias: {resultados['victorias']}")

# Probar función de evaluación
env = gym.make('FrozenLake-v1')
agente_aleatorio = AgenteAleatorio(env.action_space)
resultados = evaluar_agente(agente_aleatorio, env, num_episodios=100)
imprimir_resultados_evaluacion(resultados, "Agente Aleatorio")
env.close()



Agente Aleatorio - Resultados de Evaluación:
Tasa de Victorias: 4.0%
Recompensa Promedio: 0.040
Desviación Estándar: 0.196
Total de Victorias: 4


La siguiente celda define una función para entrenar un agente.

In [7]:
# Función de Entrenamiento de Agentes
def entrenar_agente(agente, env, num_episodios=1000, max_pasos=500, verbose=True):
    """
    Entrena un agente en el entorno.
    
    Args:
        agente: El agente a entrenar
        env: El entorno
        num_episodios: Número de episodios de entrenamiento
        max_pasos: Máximo de pasos por episodio
        verbose: Si imprimir el progreso
    
    Returns:
        list: Recompensas de episodios
    """
    recompensas_episodios = []
    longitudes_episodios = []
    num_episodios_10 = int(num_episodios / 10)
    
    for episodio in range(num_episodios):
        estado, _ = env.reset()
        recompensa_total = 0
        pasos = 0
        
        for paso in range(max_pasos):
            accion = agente.elegir_accion(estado)
            siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)

            agente.aprender(estado, accion, recompensa, siguiente_estado, terminado or truncado)
            
            estado = siguiente_estado
            recompensa_total += recompensa
            pasos += 1
            
            if terminado or truncado:
                agente.epsilon_decay_step(num_episodios)  # Decay epsilon after each episode
                break
        
        recompensas_episodios.append(recompensa_total)
        longitudes_episodios.append(pasos)
        
        if verbose and (episodio + 1) % num_episodios_10 == 0:
            recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
            longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
            print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
    
    return recompensas_episodios, longitudes_episodios

print("✓ Funciones de entrenamiento definidas")



✓ Funciones de entrenamiento definidas


La siguiente celda define el agente de Q-Learning a implementar.

## Q-Learning en entornos no deterministas

En entornos estocásticos, ejecutar una acción desde un estado no garantiza llegar siempre al mismo siguiente estado, ni recibir siempre la misma recompensa. Por eso, la actualización de la tabla Q se hace de manera **promediada**, usando un factor de aprendizaje α.

La fórmula clásica de actualización es:

$$
Q(s, a) \leftarrow (1-\alpha) Q(s, a) + \alpha \big[r + \gamma \max_{a'} Q(s', a')\big]
$$

- **α (tasa de aprendizaje):** determina cuánto peso le damos a la nueva información frente a lo aprendido hasta ahora.  
- **r + γ max Q(s', a'):** estimación de la recompensa total esperada al ejecutar la acción.
- En **entornos deterministas**, se puede sobrescribir Q(s, a) directamente porque la acción siempre lleva al mismo estado y recompensa.
- En **entornos no deterministas**, se necesita promediar la experiencia, ya que la acción puede llevar a distintos estados con distintas recompensas. El valor de Q converge así al **valor esperado a largo plazo** de cada acción.


In [8]:
class AgenteQLearning(Agente):
    """Agente que usa el algoritmo Q-Learning."""
    
    def __init__(self, espacio_estados, espacio_acciones):
        self.Q_table = np.zeros((espacio_estados.n, espacio_acciones.n))
        self.alpha = 0.05
        self.gamma = 0.95
        self.epsilon = 1
        #self.epsilon_decay = 0.995
        self.epsilon_min = 0.01
        self.espacio_acciones = espacio_acciones
        self.tam_mapa = int(np.sqrt(espacio_estados.n))  

    def elegir_accion(self, estado):
        """Elige una acción usando política epsilon-greedy.""" #o softmax, lo cambio a softmax luego 
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)  # Exploración
        else:
            return np.argmax(self.Q_table[estado])  # Explotación, accion que maximiza Q
        
    
    def aprender(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Actualiza la tabla Q usando la ecuación de Bellman."""
        max_q = np.max(self.Q_table[siguiente_estado])
        self.Q_table[estado, accion] = (1 - self.alpha) * self.Q_table[estado, accion] + self.alpha * (recompensa + self.gamma * max_q)
            
    def epsilon_decay_step(self, episodios):
        """Decae epsilon después de cada episodio."""
        self.epsilon = max(self.epsilon_min, self.epsilon - 1/episodios)
    
    def imprimirQtable(self):
        print("Tabla Q:")
        print(self.Q_table)

    def mostrar_policy(self, env):
        """
        Muestra la política aprendida en el mapa del FrozenLake.
        Flechas solo en casillas 'F' (Frozen); 'H' y 'G' se muestran tal cual.
        """
        # Mapeo de acciones a flechas: 0=izquierda, 1=abajo, 2=derecha, 3=arriba
        acciones_map = ["←", "↓", "→", "↑"]

        # Obtener el mapa original del entorno
        desc = env.unwrapped.desc.astype(str)

        # Crear copia del mapa para mostrar la política
        policy_map = desc.copy()

        # Recorrer filas y columnas del mapa
        for fila in range(self.tam_mapa):
            for col in range(self.tam_mapa):
                estado = fila * self.tam_mapa + col  # convertir fila,col a índice lineal
                if desc[fila, col] == 'F':  # solo actualizar casillas 'Frozen'
                    mejor_accion = np.argmax(self.Q_table[estado])
                    policy_map[fila, col] = acciones_map[mejor_accion]

        print("Policy aprendida:")
        for fila in policy_map:
            print(" ".join(fila))


In [9]:
# Entrenar y Evaluar el Agente Q-Learning en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)

agente = AgenteQLearning(env.observation_space, env.action_space)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=100000, verbose=False)
agente.imprimirQtable()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
agente.mostrar_policy(env)
env.close()

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG



KeyboardInterrupt: 

In [ ]:
env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
estado, _ = env.reset()
agente = AgenteQLearning(env.observation_space, env.action_space)
print(f"FrozenLake-v1 8x8: {env.render()}")
resultados, longitudes = entrenar_agente(agente, env, num_episodios=100000, verbose=True)
agente.imprimirQtable()
resultados = evaluar_agente(agente, env, num_episodios=1000)
imprimir_resultados_evaluacion(resultados, "Agente Q-Learning")
env.close()

FrozenLake-v1 8x8: 
SFFFFFFF
FFFFFFFF
FFFHFFFF
FFFFFHFF
FFFHFFFF
FHHFFFHF
FHFFHFHF
FFFHFFFG

Episodio 10000: Recompensa Promedio = 0.002, Longitud Promedio = 33.5
Episodio 20000: Recompensa Promedio = 0.007, Longitud Promedio = 36.5
Episodio 30000: Recompensa Promedio = 0.011, Longitud Promedio = 40.5
Episodio 40000: Recompensa Promedio = 0.019, Longitud Promedio = 42.7
Episodio 50000: Recompensa Promedio = 0.043, Longitud Promedio = 46.3
Episodio 60000: Recompensa Promedio = 0.070, Longitud Promedio = 49.8
Episodio 70000: Recompensa Promedio = 0.113, Longitud Promedio = 53.5
Episodio 80000: Recompensa Promedio = 0.201, Longitud Promedio = 57.6
Episodio 90000: Recompensa Promedio = 0.336, Longitud Promedio = 62.9
Episodio 100000: Recompensa Promedio = 0.551, Longitud Promedio = 69.7
Tabla Q:
[[0.04082729 0.04047659 0.04102458 0.04396887]
 [0.0455884  0.04722366 0.0505943  0.04745134]
 [0.05504945 0.05761009 0.06183769 0.05723714]
 [0.06763464 0.07152738 0.0735496  0.06878009]
 [0.08493

Aqui deberán incluir código para entrenar y evaluar el agente de Q-Learning implementado.

La siguiente celda define el agente DQN a implementar. 

In [13]:

# Detectar dispositivo
if torch.backends.mps.is_available(): 
    device = torch.device("mps")
elif torch.cuda.is_available():       
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Usando dispositivo: {device}")

class DQN(nn.Module):
    """Clase auxiliar que implementa una Red Q Profunda con una capa oculta."""
    
    def __init__(self, tamano_entrada, tamano_oculto, tamano_salida): #cant_estados, capa_oculta, cant_acciones
        super(DQN, self).__init__()
        #definicion de las capas
        self.fc1 = nn.Linear(tamano_entrada, tamano_oculto) 
        self.out = nn.Linear(tamano_oculto, tamano_salida) 
    
    def forward(self, x):
        x = torch.relu(self.fc1(x)) #funcion de activacion, opciones:  Tanh, ReLU, LeakyReLU, Sigmoid 
        return self.out(x)

class AgenteDQN(Agente):
    """Agente de Red Q Profunda."""
    
    def __init__(self, espacio_estados, espacio_acciones, capas_ocultas=32):
        self.cant_acciones = espacio_acciones
        self.cant_estados = espacio_estados
        self.alpha = 0.05
        self.gamma = 0.95
        self.epsilon = 1
        self.network_update_freq = 10  # Frecuencia de actualización de la red objetivo
        self.replay_buffer_size = 10000
        self.batch_size = 64
        self.capas_ocultas = capas_ocultas
        
    def hot_encoding_entry(self, estado):
        """Convierte un estado discreto en un vector one-hot."""
        estado_one_hot = np.zeros(self.cant_estados)
        estado_one_hot[estado] = 1
        return torch.FloatTensor(estado_one_hot).unsqueeze(0).to(device)  # Añadir dimensión de batch y mover a device

    def createModels(self):
        """Crea los modelos de política y objetivo."""
        self.policyModel = DQN(self.cant_estados, self.capas_ocultas, self.cant_acciones).to(device) #RN, que se usa para  predicer los Q-values actuales: Q(s, a) para cada acción desde un estado s
        self.targetModel = DQN(self.cant_estados, self.capas_ocultas, self.cant_acciones).to(device) #RN, se usa solo para calcular los Q-targets, 
        self.criterio = nn.MSELoss()
        self.optimizador = optim.Adam(self.policyModel.parameters(), lr=0.001)
        self.replay_buffer = deque(maxlen=self.replay_buffer_size)
        self.batch_size = self.batch_size

    def almacenar_experiencia(self, estado, accion, recompensa, siguiente_estado, terminado):
        """Almacena la experiencia en el buffer de repetición."""
        self.replay_buffer.append((estado, accion, recompensa, siguiente_estado, terminado))

    def obtener_muestra(self):
        """Muestra una experiencia aleatoria del buffer de repetición."""
        return random.sample(self.replay_buffer, self.batch_size) # retorna una lista de tuplas, de acuerdo al batch size

    def longitud_buffer(self):
        """Devuelve la longitud actual del buffer de repetición."""
        return len(self.replay_buffer)

    def elegir_accion(self, estado):
        """Elige acción usando política epsilon-greedy."""
        if np.random.rand() < self.epsilon:
            return np.random.randint(4)  # Exploración
        else:
            with torch.no_grad():
                return torch.argmax(self.policyModel(self.hot_encoding_entry(estado))).item()  # Explotación
    
    def sincronizar_modelos(self):
        """Copia los pesos del modelo de política al modelo objetivo."""
        self.targetModel.load_state_dict(self.policyModel.state_dict())

    # Función de Entrenamiento de Agentes
    def aprender(self, env, num_episodios=1000, max_pasos=500, verbose=True):

        self.sincronizar_modelos()  
        recompensas_episodios = []
        longitudes_episodios = []
        num_episodios_10 = int(num_episodios / 10)
        epsilon_history = []
        pasos_globales = 0  # Contador global de pasos para actualizar la red objetivo

        for episodio in range(num_episodios):
            estado, _ = env.reset()
            recompensa_total = 0
            pasos = 0
            
            for paso in range(max_pasos):
                accion = self.elegir_accion(estado)
                siguiente_estado, recompensa, terminado, truncado, _ = env.step(accion)
                self.almacenar_experiencia(estado, accion, recompensa, siguiente_estado, terminado or truncado)
                estado = siguiente_estado
                recompensa_total += recompensa
                pasos += 1
                pasos_globales += 1

                if terminado or truncado:
                    break
            
            recompensas_episodios.append(recompensa_total)
            longitudes_episodios.append(pasos)

            if self.longitud_buffer() > self.batch_size and np.sum(recompensas_episodios) > 0:
                mini_batch = self.obtener_muestra()
                self.optimize(mini_batch)

                # epsilon decay
                self.epsilon = max(self.epsilon - 1/num_episodios, 0)
                epsilon_history.append(self.epsilon)

                # Copiar policy a target luego de ciertos pasos indep de cant de episodios
                if pasos_globales > self.network_update_freq:
                    self.sincronizar_modelos()
                    pasos_globales = 0

            if verbose and (episodio + 1) % num_episodios_10 == 0:
                recompensa_promedio = np.mean(recompensas_episodios[-num_episodios_10:])
                longitud_promedio = np.mean(longitudes_episodios[-num_episodios_10:])
                print(f"Episodio {episodio + 1}: Recompensa Promedio = {recompensa_promedio:.3f}, Longitud Promedio = {longitud_promedio:.1f}")
        
        # Guardar el modelo entrenado
        torch.save(self.policyModel.state_dict(), "dqn_FL.pt")

        return recompensas_episodios, longitudes_episodios, epsilon_history

    def optimize(self, mini_batch):
        """Optimiza el modelo de política usando un mini-batch de experiencias."""

        #Vectorización del mini-batch
        estados = torch.cat([self.hot_encoding_entry(s) for s, _, _, _, _ in mini_batch])
        acciones = torch.tensor([a for _, a, _, _, _ in mini_batch]).unsqueeze(1) # Añadir dimensión para gather
        recompensas = torch.tensor([r for _, _, r, _, _ in mini_batch])
        siguientes_estados = torch.cat([self.hot_encoding_entry(s_) for _, _, _, s_, _ in mini_batch])
        terminados = torch.tensor([t for _, _, _, _, t in mini_batch]).float()

        # Q(s,a) actuales de la policy network (solo las acciones tomadas)
        current_q = self.policyModel(estados).gather(1, acciones) # gather para obtener los Q-values de las acciones tomadas

        # Q(s',a') máximos de la target network
        with torch.no_grad():
            max_next_q = self.targetModel(siguientes_estados).max(1)[0].unsqueeze(1) # max(1)[0] para obtener los valores máximos, unsqueeze para mantener dimensiones
            target_q = recompensas + self.gamma * max_next_q * (1 - terminados)  # Si terminado, no sumar gamma*maxQ

        # Calcular loss
        loss = self.criterio(current_q, target_q)

        # Optimización
        self.optimizador.zero_grad()
        loss.backward()
        self.optimizador.step()

    def test_model(self, env, num_episodios=1000, trained_model_path="dqn_FL.pt"):
        """Función para testear el modelo cargado."""
        policy_dqn = DQN(self.cant_estados, self.capas_ocultas, self.cant_acciones) 
        policy_dqn.load_state_dict(torch.load(trained_model_path))
        policy_dqn.eval()  

        print('Policy (trained):')

        recompensas_totales = []
        victorias = 0
        for episodio in range(num_episodios):
            estado, _ = env.reset()
            recompensa_total = 0
            while True:
                estado_tensor = self.hot_encoding_entry(estado)
                with torch.no_grad():
                    accion = torch.argmax(policy_dqn(estado_tensor)).item()
                estado, recompensa, terminado, truncado, _ = env.step(accion)
                recompensa_total += recompensa
                if terminado or truncado:
                    break
            recompensas_totales.append(recompensa_total)
            if recompensa_total > 0:
                victorias += 1
        tasa_victorias = victorias / num_episodios
        recompensa_promedio = np.mean(recompensas_totales)
        desv_estandar = np.std(recompensas_totales)
        print(f'Tasa de victorias: {tasa_victorias:.1%}, Recompensa Promedio: {recompensa_promedio:.3f}, Desviación Estándar: {desv_estandar:.3f}')

    


Usando dispositivo: cpu


Aqui deberán incluir código para entrenar y evaluar el agente de DQN 
implementado.


In [16]:
# Entrenar y Evaluar el Agente Q-Learning en FrozenLake-v1 4x4 
env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=True, render_mode="ansi",  max_episode_steps=500)
agente = AgenteDQN(env.observation_space.n, env.action_space.n, capas_ocultas=64)
estado, _ = env.reset()
print(f"FrozenLake-v1 4x4: {env.render()}")
agente.createModels()
resultados, longitudes, eps_history = agente.aprender(env, num_episodios=100000, verbose=True)

FrozenLake-v1 4x4: 
SFFF
FHFH
FFFH
HFFG

Episodio 10000: Recompensa Promedio = 0.015, Longitud Promedio = 7.7
Episodio 20000: Recompensa Promedio = 0.018, Longitud Promedio = 7.2
Episodio 30000: Recompensa Promedio = 0.017, Longitud Promedio = 6.8
Episodio 40000: Recompensa Promedio = 0.019, Longitud Promedio = 6.6
Episodio 50000: Recompensa Promedio = 0.021, Longitud Promedio = 6.3
Episodio 60000: Recompensa Promedio = 0.022, Longitud Promedio = 6.2
Episodio 70000: Recompensa Promedio = 0.023, Longitud Promedio = 6.0
Episodio 80000: Recompensa Promedio = 0.029, Longitud Promedio = 6.0
Episodio 90000: Recompensa Promedio = 0.028, Longitud Promedio = 5.9
Episodio 100000: Recompensa Promedio = 0.035, Longitud Promedio = 6.8


In [15]:
agente.test_model(env, num_episodios=1000)
env.close()

Policy (trained):
Tasa de victorias: 6.1%, Recompensa Promedio: 0.061, Desviación Estándar: 0.239


In [ ]:
import torch
# Detectar dispositivo
if torch.backends.mps.is_available(): 
    device = torch.device("mps")
elif torch.cuda.is_available():       
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Usando dispositivo: {device}")

Usando dispositivo: cpu
